<div style="padding: 20px; background: linear-gradient(90deg, #8E2DE2 0%, #4A00E0 100%); border-radius: 10px; color: white;">
    <h1 style="color: white; border-bottom: none;">🦸‍♂️ Module 7.2: HyDE</h1>
    <p style="font-size: 1.2em; opacity: 0.9;">Hypothetical Document Embeddings - Bridging the Semantic Gap.</p>
</div>

---

## 1. The "Semantic Gap" Problem

Vector embeddings map text to multi-dimensional space based on meaning and structure. 
A short user query like `"how do bees communicate"` is a **Question**.
A document that holds the answer is an **Explanation**: `"Apis mellifera uses a complex waggle dance consisting of figure-eight movements..."`

Because a Question is structurally and semantically different from a detailed Explanation, their vectors might actually be far apart in the vector space! This is the Semantic Gap.

## 2. The HyDE Solution
HyDE solves this with a brilliant trick:
1. Take the user's Question and ask an LLM to answer it without using the Vector DB.
2. The LLM hallucinates a **Hypothetical Answer**.
3. Instead of embedding the user's Question, we **embed the LLM's Hypothetical Answer**!
4. We search the DB. Because the Hypothetical Answer is an Explanation, it perfectly matches the structure of the real Explanations in the database!

In [1]:
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from dotenv import load_dotenv
import os
import warnings

warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
load_dotenv()

docs = [
    Document(page_content="BERT uses bidirectional context to understand language, trained with masked language modelling."),
    Document(page_content="GPT-4 is a decoder-only autoregressive model that predicts the next token."),
    Document(page_content="T5 treats every NLP task as a text-to-text problem."),
    Document(page_content="Llama 3 is Meta's open-weights model family available in 8B and 70B sizes."),
    Document(page_content="RoBERTa improves on BERT by training longer with more data and removing the next sentence prediction task."),
]

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vs = Chroma.from_documents(docs, embeddings, collection_name="hyde_demo")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## 3. Implementing HyDE
We use Groq to quickly hallucinate a response, then embed it.

In [2]:
groq_api_key = os.environ.get("GROQ_API_KEY")

if groq_api_key:
    llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.5)
    
    hyde_prompt = ChatPromptTemplate.from_template("""
    Write a short, factual paragraph (2-3 sentences) that would answer this question.
    Do NOT say you don't know — write a plausible, detailed response.
    
    Question: {question}
    
    Hypothetical answer:
    """)
    
    def hyde_retrieve(question: str, k: int = 3):
        # 1. Hallucinate the document
        chain = hyde_prompt | llm | StrOutputParser()
        hyp_doc = chain.invoke({"question": question})
        
        # 2. Embed the hallucinatory text
        hyp_vec = embeddings.embed_query(hyp_doc)
        
        # 3. Search the Vector DB using the hallucinatory vector
        results = vs.similarity_search_by_vector(hyp_vec, k=k)
        return hyp_doc, results
        
    query = "How does BERT understand language context?"
    print(f"QUERY: {query}\n")
    
    # --- Standard Retrieval ---
    std_results = vs.similarity_search(query, k=2)
    print("--- STANDARD RETRIEVAL (Using Query Vector) ---")
    for d in std_results:
        print(f" • {d.page_content[:80]}")
        
    # --- HyDE Retrieval ---
    hyp_doc, hyde_results = hyde_retrieve(query, k=2)
    print("\n--- HYDE RETRIEVAL ---")
    print(f"[LLM Hallucination]: {hyp_doc.strip()}")
    print("\n[Retrieved using Hallucination Vector]:")
    for d in hyde_results:
        print(f" • {d.page_content[:80]}")
else:
    print("GROQ_API_KEY missing. Please add to .env file.")

QUERY: How does BERT understand language context?

--- STANDARD RETRIEVAL (Using Query Vector) ---
 • BERT uses bidirectional context to understand language, trained with masked lang
 • RoBERTa improves on BERT by training longer with more data and removing the next



--- HYDE RETRIEVAL ---
[LLM Hallucination]: BERT (Bidirectional Encoder Representations from Transformers) understands language context through its bidirectional encoder architecture, which processes input text from both the left and right sides simultaneously. This allows BERT to capture contextual relationships between words and phrases by encoding the entire input sequence into a continuous vector representation, known as a contextualized embedding. By analyzing these embeddings, BERT can identify nuances in language, such as sentiment, intent, and entity relationships, enabling it to better comprehend the context of a given text.

[Retrieved using Hallucination Vector]:
 • BERT uses bidirectional context to understand language, trained with masked lang
 • RoBERTa improves on BERT by training longer with more data and removing the next
